# MSE590500 Homework assignment 4-5

## Due on June 18 (Thursday), 2026

This is our last homework assignment. There are three problem sets, including 1) finite difference method for binary diffusion, 2) calculating average grain size, and 3) identify phase boundaries. **Please complete and submit it before the dealine such that it can be included into your final grades.**

#### <p style="text-align: right;"> &#9989; **put your name here** </p>

---
## Problem 1. Simulation of binary diffusion in 2D. (30 pt)

In a 2D domain, the dimensions are 130 and 42 for width (horizontal) and height (vertical), respectively. There are two materials sources on the north boundary, with a width of 16, see the red and blue line segments in the figure below. The two segments are 31 and 80 away from the west boundary, respectively. 

<img src="https://i.ibb.co/4w320rLR/2-D-diff-2.jpg" width="500">

Materials A and B are diffusing into the 2D region from the red and blue segments, respectively. You are asked to simulate the diffusion process of the two species. The governing equations for the two diffusions are

$$\frac{\partial C_A}{\partial t} = D_A \nabla^2 C_A - r_A C_A C_B,$$

$$\frac{\partial C_B}{\partial t} = D_B \nabla^2 C_B - r_B C_A C_B,$$

where $C_i$ are concentrations, $D_i$ are diffusion coefficients , and $r_i$ are the recombining rates of the $i$-th species. The recombination means when $A$ and $B$ meet together, they annihilate. 

### Discretization:

We will discretize the domain with uniform grid spacings. 130 grids and 42 grids in the width and height directions, respectively, i.e., $\Delta x=1$.
As such, the indice along the width direction, $j$, go from 0 to 129, and $i$ along the height direction go from 0 to 41.


### Stencils: 

The finite difference method stencils are 

$$\frac{C_{A(i,j)}^{(n+1)}-C_{A(i,j)}^{(n)}}{\Delta t} = D_A \frac{C_{A(i-1,j)}^{(n)}+C_{A(i+1,j)}^{(n)}+C_{A(i,j-1)}^{(n)}+C_{A(i,j+1)}^{(n)}-4C_{A(i,j)}^{(n)}}{\Delta x^2} - r_A C_{A(i,j)}^{(n)}C_{B(i,j)}^{(n)},$$


$$\frac{C_{B(i,j)}^{(n+1)}-C_{B(i,j)}^{(n)}}{\Delta t} = D_B \frac{C_{B(i-1,j)}^{(n)}+C_{B(i+1,j)}^{(n)}+C_{B(i,j-1)}^{(n)}+C_{B(i,j+1)}^{(n)}-4C_{B(i,j)}^{(n)}}{\Delta x^2} - r_B C_{A(i,j)}^{(n)}C_{B(i,j)}^{(n)},$$

where subscripts $i$ and $j$ are the indices of grid points, superscripts $(n)$ denote time steps, $\Delta x$ is the grid spacing, and $\Delta t $ is the time step size. Thus, the concentration of $A$ at grid point $(i,j)$ is updated by

$$C_{A(i,j)}^{(n+1)} = C_{A(i,j)}^{(n)} + \Delta t \times \bigg( D_A \frac{C_{A(i-1,j)}^{(n)}+C_{A(i+1,j)}^{(n)}+C_{A(i,j-1)}^{(n)}+C_{A(i,j+1)}^{(n)}-4C_{A(i,j)}^{(n)}}{\Delta x^2} - r_A C_{A(i,j)}^{(n)}C_{B(i,j)}^{(n)} \bigg);$$

similar method for updating $C_B$.

### Boundary conditions:

We solve the concentrations only within the domain. Thus, values on the domain boundaries are calculated. Here, we assume no-flux boundary conditions on all box boundaries except for the two materials sources (red and blue segments). For no-flux boundary condition on the west boundary, we have

$$\frac{\partial C}{\partial x}\bigg|_w = 0 ~~\Longrightarrow~~\frac{C_{i,0}-C_{i,1}}{\Delta x} = 0 ~~\Longrightarrow~~ C_{i,0} = C_{i,1}.$$

Thus, to impose such boundary conditons, we just need to simply set `C[:,0] = C[:,1]`. Similarly, for the east boundary, we set `C[:,-1] = C[:,-2]`, after every time stepping. On the south side: `C[0,:] = C[1,:]`. On the north side: `C[-1,:] = C[-2,:]` except for the regions of materials sources. 

At the materials sources, the source concentrations are fixed: $C_A = 1$ in the red segment region, and $C_B = 0.75$ in the blue segment region, i.e., `CA[-1,31:47] = 1` and `CB[-1,80:96] = 0.75`.


### Materials parameters:

The materials parameters are $D_A=0.8$, $D_B=1$, $r_A=0.4$, and $r_B = 0.2$. Time step size is $\Delta t= 0.05$.


#### Complete the code below to run the simulation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import time

# complete the code, fill up ??

# grid
px = 130
py = 42

CA = np.zeros((py, px))
CB = np.zeros((py, px))

# parameters
DA = ??
DB = ??
rA = ??
rB = ??
dx = ??
dt = ??

# initial boundary conditions
CA[??, ??] = 1.0
CB[??, ??] = 0.75

# plotting: create figure only once
fig, axes = plt.subplots(1, 2, figsize=(8, 2))

im0 = axes[0].imshow(CA, cmap="jet", vmin=0, vmax=1, origin="lower")
axes[0].set_title("CA")
fig.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(CB, cmap="jet", vmin=0, vmax=1, origin="lower")
axes[1].set_title("CB")
fig.colorbar(im1, ax=axes[1])

plt.tight_layout()

# display once
display(fig)



# time evolution
for t in range(40000):

    CA_new = CA.copy()
    CB_new = CB.copy()

    lap_CA = (CA[??,??] + CA[??,??] + CA[??,??] + CA[??,??] 
              - 4 * CA[??,??]) / dx**2

    lap_CB = (CB[??,??] + CB[??,??] + CB[??,??] + CB[??,??]
              - 4 * CB[??,??]) / dx**2

    recombA = rA * CA[1:-1, 1:-1] * CB[1:-1, 1:-1]
    recombB = rB * CA[1:-1, 1:-1] * CB[1:-1, 1:-1]

    CA_new[1:-1, 1:-1] += (DA * ?? - ??) * dt
    CB_new[1:-1, 1:-1] += (DB * ?? - ??) * dt

    CA = CA_new.copy()
    CB = CB_new.copy()

    # boundary conditions
    CA[0, :] = CA[??, :]
    CA[-1, ??] = CA[-2, :]
    CA[:, 0] = CA[:, ??]
    CA[:, ??] = CA[:, -2]
    CA[-1, 31:47] = 1.0

    CB[0, :] = CB[??,??]
    CB[-1, :] = CB[??,??]
    CB[:, 0] = CB[??,??]
    CB[:, -1] = CB[??,??]
    CB[-1, 80:96] = ??

    # visualization
    if t % 200 == 1:

        im0.set_data(CA)
        im1.set_data(CB)

        axes[0].set_title(f"CA, step={t}")
        axes[1].set_title(f"CB, step={t}")

        clear_output(wait=True)
        display(fig)

        time.sleep(0.0002)   

---
## Problem 2. Calculate averge grain size. (30 pt)

Below is a snythetic SEM image with EBSD determined grain orientations, generated by GPT. You are asked to write a code to calculate the average grain size.

<img src="https://i.ibb.co/DPb1Kgmb/Grains2.png" width="500">

- Download the image and load in the image.
`https://raw.githubusercontent.com/huichiayu/cmse_202_802/main/IMGs/Grains2.png`
- Plot and examine the image.

In [ ]:
# put your code here
import numpy as np
import matplotlib.pyplot as plt
from imageio.v2 import imread

img = ??

plt.figure(figsize=(5,5))
plt.imshow(img, cmap='gray')
plt.axis('off')
plt.show()

print(img.shape)

- Convert this image a gray-scale one.

In [ ]:
# put your code here

g_img = ??
print(g_img.shape)

plt.figure(figsize=(5,5))
plt.imshow(g_img, cmap="gray")
plt.axis('off')
plt.show()

- Use the thresholding code in the previous coding assignments to segment the gray-scale image to binary image. Find a good set of `vmin` and `vmax`. You are expecting something like this.

<img src="https://i.ibb.co/HTskGhwG/Grain-g.png" width="500">

In [ ]:
# put your code here
from ipywidgets import interact, fixed
import matplotlib.pyplot as plt

def gray_threshold(im, vmin=0.0, vmax=1.0, s=True):

    b_img = (im > vmin) & (im < vmax)

    print("vmin =", vmin)
    print("vmax =", vmax)

    if s:
        plt.figure(figsize=(12,5))
    
        plt.subplot(1,2,1)
        plt.imshow(im, cmap='gray')
        plt.title("Original")
        plt.axis("off")
    
        plt.subplot(1,2,2)
        plt.imshow(b_img, cmap='gray')
        plt.title("Binary segmentation")
        plt.axis("off")
    
        plt.show()

    return b_img

- Run the cell below to find a good set to `vmin` and `vmax`.

In [ ]:
w = interact(
    gray_threshold,
    im=fixed(g_img),
    vmin=(0.0, 1.0, 0.01),
    vmax=(0.0, 1.0, 0.01),
    __manual=True)

- Use `erosion` and `dilation` in `ndimage` of `scipy` to highlight the grain boundary areas, for example, as in the figure below. You should try different iterations of erosion and dilation. In this case, grains are separated more. 

<img src="https://i.ibb.co/5WPYctP9/binary.png" width="500">


In [ ]:
# put your code here
from scipy import misc, ndimage

b_img = gray_threshold(g_img, ??)

after_erosion = ndimage.??
after_dialation = ndimage.??

plt.imshow(after_dialation, cmap="gray")
plt.show() 

- Use `label` in `ndimage` to count how many separate regions (grains) in the image you obtained.
- Divide the total area of the image to the number of grains.
- Then, use

$$ r_{eff} = \sqrt{A/\pi}$$

to calculate the effect grain radius.

In [ ]:
# put your code here
from scipy.ndimage import label

lab, num_features = ??

plt.figure(figsize=(5,4))
plt.imshow(lab)
plt.colorbar()

n_grains = ??
print(n_grains)

A_tot = ??
r_eff = ??
r_eff

---
## Problem 3. Phase boundaries. (30 pt)

Here, you are asked to use SVM classifier to delineate phases boundaries on a ternay phase diagarm. You are expecting a result like the figure below.


<img src="https://i.ibb.co/BHx5SX1C/ternary-pd.png" width="500">


- Download the dataset from `https://raw.githubusercontent.com/huichiayu/cmse_202_802/main/data/synthetic_ternary_phase_data.csv`. It contains 120 data point, each point contains the composition of three components and a label of its corresponding phase. For instance,
  
<img src="https://i.ibb.co/3m9SLyMH/data-ternary.png" width="360">
 
- Read the csv file using `pandas`. 


In [ ]:
# !Curl -O ??

In [ ]:
# put your code
import pandas as pd

Data=pd.read_csv(??)
Data = Data.dropna()
Data.head()

- Plot these data points a ternary composition space. You are expecting something like the figure below.
  
<img src="https://i.ibb.co/NdJH3s18/Ternery-Points.png" width="400">

- Convert ternary points to Cartesian coordinate for plotting.

In [ ]:
# convert
import numpy as np

def ternary_to_xy(xA, xB, xC):
    x = ??
    y = ??
    return x, y

xA = Data["xA"].values
xB = Data["xB"].values
xC = Data["xC"].values

Xplot, Yplot = ternary_to_xy(xA, xB, xC)


- To assign colors, we need to convert string labels to numeric labels, i.e., A, B, C, ABC to 0, 1 , 2, 3, respectively.

In [ ]:
import matplotlib.pyplot as plt

labels = Data["phase"]

phase_to_int = {
    "A":0,
    "B":1,
    "C":2,
    "ABC":3}

colors = [phase_to_int[p] for ?? in ??]

plt.figure(figsize=(7,6))

plt.scatter(
    Xplot,
    Yplot,
    c=colors,
    s=10)

plt.axis('equal')
plt.show()

- Make a feature array from the composition data (i.e., xA, xB, xC).
- Make a output vector from the phase data. Note that, for SVM library, we need to numeric labels, i.e., 0, 1, 2, 3.

In [ ]:
X_features = ??
y = ??


- Create train and test data using 0.25 of them as the test data. 

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split( ?? )

- Scale the features 
- Build a SVM classifier model
- Train the model
- Evaluate the model performance


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix

# scale features
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# SVM classifier
svm_model = SVC(??)

svm_model.fit(??, ??)

y_pred = svm_model.predict(??)

print(classification_report(??, ??))
print(confusion_matrix(??, ??))

### Visualization

- Set up ternary grids uniformly over the composition space
- Use the SVM classifier for these grid points
- Plot the predicted label values using `tricontour`. Note that `tricontour` needs coordinates ($x,y$). Thus, you need to convert ternary composition to Cartesian for plotting.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# =====================================================
# 1. Create dense ternary grid
# =====================================================

def make_ternary_grid(n=250):
    pts = []

    for i in range(n + 1):
        for j in range(n + 1 - i):

            xA = i / n
            xB = j / n
            xC = 1.0 - xA - xB

            pts.append([xA, xB, xC])

    return np.array(pts)


grid = make_ternary_grid(n=250)

# =====================================================
# 2. Predict phase labels on ternary grid
# =====================================================

grid_pred = svm_model.predict(??)

# =====================================================
# 3. Convert ternary coordinates to 2D Cartesian
# =====================================================

xg, yg = ternary_to_xy(??, ??, ??)

# =====================================================
# 4. Plot predicted phase regions
# =====================================================

plt.figure(figsize=(7, 6))

plt.tricontourf(
    ??,
    ??,
    grid_pred,
    levels= ??,
    alpha=0.45)

# draw phase boundaries
plt.tricontour(
    xg,
    yg,
    grid_pred,
    levels=np.arange(grid_pred.min() + 0.5, grid_pred.max() + 0.5),
    colors="k",
    linewidths=1.5)



# =====================================================
# Annotate phase names
# =====================================================

phase_names = {
    0: "A",
    1: "B",
    2: "C",
    3: "ABC co-exist"}

for phase_id, phase_name in phase_names.items():

    mask = (grid_pred == phase_id)

    if np.sum(mask) == 0:
        continue

    x_center = np.mean(xg[mask])
    y_center = np.mean(yg[mask])

    plt.text(
        x_center,
        y_center,
        phase_name,
        fontsize=14,
        fontweight='bold',
        ha='center',
        va='center',
        bbox=dict(
            facecolor='white',
            alpha=0.7,
            edgecolor='none'
        )
    )

plt.axis("equal")
plt.axis("off")
plt.title("SVM-Predicted Ternary Phase Regions")

plt.show()

---
## Problem 4. Survery questionnaire (10 pt)

### Lastly, Please complete this [**survey**](https://msu.co1.qualtrics.com/jfe/form/SV_01bGLQLo78tifH0).

---
# Congradulations! You're done. Please upload you completed file via the [**LINK**](https://www.dropbox.com/request/v0lou94ybz28k8huexh5).